In [1]:
# =====================================================
# MÓDULO 1
# Predicción del Riesgo de Diabetes
# Dataset: CDC Diabetes Health Indicators
# Algoritmos:
#   - Logistic Regression (class_weight="balanced")
#   - Random Forest (class_weight="balanced")
#
# Fix aplicado (revisión backend, 2026-07-25):
#   El dataset esta desbalanceado (~86% / 14%). Sin compensar el
#   desbalance, ambos modelos daban Recall < 0.17 (detectaban casi
#   ningun caso de riesgo real). Se agrega class_weight="balanced"
#   a los dos algoritmos, que es lo que exige la prioridad de Recall
#   definida en la Fase 5 del documento. Se exporta el pipeline
#   ganador (mayor Recall) a ../models_artifacts/ para que el backend
#   lo cargue en produccion.
# =====================================================

import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# =========================
# Cargar dataset
# =========================

df = pd.read_csv("../data/diabetes_binary_health_indicators_BRFSS2015.csv")

# Eliminar registros duplicados
df = df.drop_duplicates()

print("Primeras filas")
print(df.head())

print("\nInformación")
print(df.info())

print("\nValores nulos")
print(df.isnull().sum())

# =========================
# Separar variables
# =========================

X = df.drop("Diabetes_binary", axis=1)
y = df["Diabetes_binary"]

num_cols = X.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X.select_dtypes(include=["object"]).columns

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols)
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# =====================================================
# MODELO 1 - REGRESIÓN LOGÍSTICA (balanceada)
# =====================================================

modelo_lr = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("modelo", LogisticRegression(max_iter=1000, class_weight="balanced"))
])

modelo_lr.fit(X_train, y_train)
pred_lr = modelo_lr.predict(X_test)
prob_lr = modelo_lr.predict_proba(X_test)[:, 1]

print("\n========================================")
print("RESULTADOS - REGRESIÓN LOGÍSTICA (balanced)")
print("========================================")
print("Accuracy :", round(accuracy_score(y_test, pred_lr), 4))
print("Precision:", round(precision_score(y_test, pred_lr), 4))
print("Recall   :", round(recall_score(y_test, pred_lr), 4))
print("F1 Score :", round(f1_score(y_test, pred_lr), 4))
print("ROC AUC  :", round(roc_auc_score(y_test, prob_lr), 4))
print("\nMatriz de Confusión")
print(confusion_matrix(y_test, pred_lr))
print("\nClassification Report")
print(classification_report(y_test, pred_lr))

# =====================================================
# MODELO 2 - RANDOM FOREST (balanceado)
# =====================================================

modelo_rf = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("modelo", RandomForestClassifier(
        n_estimators=200, random_state=42, class_weight="balanced"
    ))
])

modelo_rf.fit(X_train, y_train)
pred_rf = modelo_rf.predict(X_test)
prob_rf = modelo_rf.predict_proba(X_test)[:, 1]

print("\n========================================")
print("RESULTADOS - RANDOM FOREST (balanced)")
print("========================================")
print("Accuracy :", round(accuracy_score(y_test, pred_rf), 4))
print("Precision:", round(precision_score(y_test, pred_rf), 4))
print("Recall   :", round(recall_score(y_test, pred_rf), 4))
print("F1 Score :", round(f1_score(y_test, pred_rf), 4))
print("ROC AUC  :", round(roc_auc_score(y_test, prob_rf), 4))
print("\nMatriz de Confusión")
print(confusion_matrix(y_test, pred_rf))
print("\nClassification Report")
print(classification_report(y_test, pred_rf))

# =====================================================
# COMPARACIÓN FINAL Y SELECCIÓN DEL GANADOR
# =====================================================

resultados = pd.DataFrame({
    "Modelo": ["Logistic Regression", "Random Forest"],
    "Accuracy": [accuracy_score(y_test, pred_lr), accuracy_score(y_test, pred_rf)],
    "Precision": [precision_score(y_test, pred_lr), precision_score(y_test, pred_rf)],
    "Recall": [recall_score(y_test, pred_lr), recall_score(y_test, pred_rf)],
    "F1": [f1_score(y_test, pred_lr), f1_score(y_test, pred_rf)],
    "ROC_AUC": [roc_auc_score(y_test, prob_lr), roc_auc_score(y_test, prob_rf)],
})

print("\n========================================")
print("COMPARACIÓN DE MODELOS")
print("========================================")
print(resultados)

# Se prioriza Recall (Fase 5): elegir el modelo con mayor Recall.
candidatos = {"Logistic Regression": modelo_lr, "Random Forest": modelo_rf}
ganador_nombre = resultados.sort_values("Recall", ascending=False).iloc[0]["Modelo"]
ganador_pipeline = candidatos[ganador_nombre]

print(f"\nModelo ganador por Recall: {ganador_nombre}")

joblib.dump(ganador_pipeline, "../models_artifacts/modelo1_riesgo_diabetes.joblib")
print("Guardado en ../models_artifacts/modelo1_riesgo_diabetes.joblib")
print("Columnas esperadas por el modelo:", list(X.columns))


Primeras filas
   Diabetes_binary  HighBP  HighChol  CholCheck   BMI  Smoker  Stroke  \
0              0.0     1.0       1.0        1.0  40.0     1.0     0.0   
1              0.0     0.0       0.0        0.0  25.0     1.0     0.0   
2              0.0     1.0       1.0        1.0  28.0     0.0     0.0   
3              0.0     1.0       0.0        1.0  27.0     0.0     0.0   
4              0.0     1.0       1.0        1.0  24.0     0.0     0.0   

   HeartDiseaseorAttack  PhysActivity  Fruits  ...  AnyHealthcare  \
0                   0.0           0.0     0.0  ...            1.0   
1                   0.0           1.0     0.0  ...            0.0   
2                   0.0           0.0     1.0  ...            1.0   
3                   0.0           1.0     1.0  ...            1.0   
4                   0.0           1.0     1.0  ...            1.0   

   NoDocbcCost  GenHlth  MentHlth  PhysHlth  DiffWalk  Sex   Age  Education  \
0          0.0      5.0      18.0      15.0       1.


RESULTADOS - REGRESIÓN LOGÍSTICA (balanced)
Accuracy : 0.7143
Precision: 0.3183
Recall   : 0.7601
F1 Score : 0.4487
ROC AUC  : 0.8106

Matriz de Confusión
[[27448 11428]
 [ 1684  5335]]

Classification Report
              precision    recall  f1-score   support

         0.0       0.94      0.71      0.81     38876
         1.0       0.32      0.76      0.45      7019

    accuracy                           0.71     45895
   macro avg       0.63      0.73      0.63     45895
weighted avg       0.85      0.71      0.75     45895




RESULTADOS - RANDOM FOREST (balanced)
Accuracy : 0.8103
Precision: 0.3923
Recall   : 0.4374
F1 Score : 0.4136
ROC AUC  : 0.7844

Matriz de Confusión
[[34121  4755]
 [ 3949  3070]]

Classification Report
              precision    recall  f1-score   support

         0.0       0.90      0.88      0.89     38876
         1.0       0.39      0.44      0.41      7019

    accuracy                           0.81     45895
   macro avg       0.64      0.66      0.65     45895
weighted avg       0.82      0.81      0.81     45895




COMPARACIÓN DE MODELOS
                Modelo  Accuracy  Precision    Recall        F1   ROC_AUC
0  Logistic Regression  0.714304   0.318260  0.760080  0.448659  0.810583
1        Random Forest  0.810350   0.392332  0.437384  0.413635  0.784406

Modelo ganador por Recall: Logistic Regression
Guardado en ../models_artifacts/modelo1_riesgo_diabetes.joblib
Columnas esperadas por el modelo: ['HighBP', 'HighChol', 'CholCheck', 'BMI', 'Smoker', 'Stroke', 'HeartDiseaseorAttack', 'PhysActivity', 'Fruits', 'Veggies', 'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost', 'GenHlth', 'MentHlth', 'PhysHlth', 'DiffWalk', 'Sex', 'Age', 'Education', 'Income']
